The goal here is to develop and evaluate machine learning and deep learning models for product price prediction using the full preprocessed dataset hosted on Hugging Face, rather than the lite version. Baseline and other traditional machine learning models are trained and evaluated to establish reference performance.

Deep learning models, including an Artificial Neural Network (ANN) and a Stacked Autoencoder (SAE)-based regression model, are implemented to explore complex relationships between product summaries and their corresponding prices. Model performance is evaluated and compared to assess predictive accuracy and the effectiveness of the different modeling methods.

Mount google drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


Verify Colab can retrieve your GitHub secret.

In [ ]:
from google.colab import userdata
github_token = userdata.get("GITHUB_TOKEN")
print("Github token added", github_token is not None)

Github token added True


In [ ]:
import os
os.chdir("/content")
#Clone repo after a new Colab runtime as the previous will be erased
!git clone https://github.com/Don4141/ProductPriceAI.git

#Set repository as working directory
os.chdir("/content/ProductPriceAI")

#Verify
print("Repo exist", os.path.exists("/content/ProductPriceAI"))
print("Current working directory", os.getcwd())

Cloning into 'ProductPriceAI'...
remote: Enumerating objects: 139, done.
remote: Counting objects: 100% (139/139), done.
remote: Compressing objects: 100% (109/109), done.
remote: Total 139 (delta 62), reused 78 (delta 21), pack-reused 0 (from 0)
Receiving objects: 100% (139/139), 14.84 MiB | 18.57 MiB/s, done.
Resolving deltas: 100% (62/62), done.
Repo exist True
Current working directory /content/ProductPriceAI


In [ ]:
# Load API credentials
for key in ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GROQ_API_KEY"]:
  os.environ[key] = userdata.get(key)

In [ ]:
 #Verify if API keys are loaded
for key in ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GROQ_API_KEY"]:
    print(f"{key}: {'loaded' if os.getenv(key) else 'Missing'}")

OPENAI_API_KEY: loaded
ANTHROPIC_API_KEY: loaded
GROQ_API_KEY: loaded


In [ ]:
#Import project modules
import random
import pandas as pd
import numpy as np
import stat
import copy
import math
from pricer.items import Item
from dotenv import load_dotenv
from huggingface_hub import login
import xgboost as xgb
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from torch.utils.data import Dataset, DataLoader, DataLoader, TensorDataset
from torch.optim.lr_scheduler import CosineAnnealingLR
import torch.nn as nn
import torch.optim as optim
from pricer.evaluator import evaluate, Tester
#from litellm import completion
from tqdm.auto import tqdm
from tqdm.notebook import tqdm
import csv
import torch

Load the preprocessed dataset containing 819,998 products from Hugging Face

In [ ]:
dataset = f"{"sadjei65320"}/items_full"
train, val, test = Item.from_hub(dataset)
items = train + val + test

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

README.md:   0%|          | 0.00/744 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  237MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.00MB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.00MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/799998 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Loaded 799,998 training items, 10,000 validation items, 10,000 test items


Establish a reference against which the trained models can be compared

In [ ]:
#Define a random baseline that predicts product prices between $0 and $999.
#This baseline model just takes an item and estimate its price without considering any product features in the prediction.
def random_pricer(item):
    return random.randrange(1,1000)

In [ ]:
#Evaluate the performance of the function on the first 1000 items in the test data.
#The evaluate() function iterates through the test products, obtains predictions for 1000 of the products, and compares them with their actual prices.
random.seed(42)
random_pricer_tester = evaluate(random_pricer, test, size=1000)

  0%|          | 0/1000 [00:00<?, ?it/s]

$596 $95 $352 $688 $104 $687 $719 $142 $104 $119 $414 $550 $513 $52 $184 $584 $549 $214 $898 $398 $268 $542 $89 $60 $24 $190 $494 $499 $8 $540 $124 $684 $641 $589 $411 $20 $0 $86 $304 $145 $699 $428 $226 $743 $781 $83 $655 $386 $174 $258 $139 $208 $501 $629 $84 $85 $25 $360 $82 $129 $654 $266 $537 $227 $795 $36 $721 $451 $470 $78 $982 $761 $338 $46 $326 $272 $745 $628 $606 $867 $865 $291 $471 $7 $582 $28 $32 $588 $1 $746 $280 $47 $834 $215 $828 $49 $90 $215 $405 $591 $844 $185 $57 $296 $184 $112 $672 $66 $563 $861 $620 $574 $38 $568 $91 $361 $497 $593 $154 $140 $334 $86 $253 $838 $516 $662 $512 $135 $686 $318 $689 $771 $513 $2 $182 $815 $57 $800 $122 $226 $145 $127 $120 $853 $936 $548 $875 $650 $188 $200 $642 $392 $306 $848 $837 $549 $375 $123 $244 $118 $326 $747 $427 $533 $119 $564 $837 $744 $277 $564 $347 $352 $85 $27 $444 $205 $26 $674 $141 $826 $127 $137 $175 $138 $792 $452 $378 $383 $145 $362 $268 $571 $463 $382 $234 $896 $518 $773 $936 $16 $632 $394 $106 $669 $582 $429 $689 $75 $

In [ ]:
#Generate a performance plot
random_pricer_tester.report()

The two performance evaluation plots illustrate prediction accuracy and how the average prediction error changes across 1,000 test products.

The MAE plot shows how the cumulative average absolute prediction error changes as more test products are evaluated. The red line represents the running MAE, while the shaded region represents the estimated 95% confidence interval around the mean. The MAE stabilizes at \$399.26 ± \$16.99 (95% CI) after 1,000 predictions, indicating consistently large prediction errors.


The scatter plot compares actual product prices against randomly predicted prices, with the diagonal dashed line representing perfect predictions. Most predictions fall far from the diagonal, indicating poor predictive accuracy, particularly for lower-priced products. The model achieves MAE = \$399.26, MSE = 234,592, and R² ≈ -8.448 (-844.8%), demonstrating that random predictions perform substantially worse than predicting the mean actual price.

----

In [ ]:
#Calculate the mean training price to establish a constant price baseline
train_prices = [item.price for item in train]
train_average = sum(train_prices) / len(train_prices)
print(f"{train_average:.2f}")

143.96


In [ ]:
def constant_pricer(item):
    return train_average

In [ ]:
constant_pricer_tester = evaluate(constant_pricer, test, size=1000)

  0%|          | 0/1000 [00:00<?, ?it/s]

$85 $124 $234 $72 $3 $825 $108 $57 $65 $120 $135 $220 $98 $106 $431 $129 $88 $81 $122 $56 $21 $431 $24 $108 $56 $285 $120 $25 $108 $109 $64 $94 $119 $14 $5 $306 $82 $230 $156 $4 $14 $319 $89 $109 $99 $103 $84 $97 $31 $117 $123 $131 $336 $9 $117 $124 $74 $114 $126 $95 $70 $57 $62 $100 $112 $63 $117 $124 $64 $94 $128 $40 $94 $109 $96 $115 $39 $128 $116 $104 $126 $64 $23 $46 $4 $100 $129 $54 $91 $98 $127 $109 $102 $120 $84 $89 $336 $74 $84 $84 $132 $45 $34 $60 $36 $41 $129 $64 $13 $45 $64 $54 $108 $88 $416 $393 $94 $10 $261 $116 $4 $331 $120 $34 $4 $101 $85 $216 $128 $129 $31 $128 $138 $88 $91 $117 $54 $119 $58 $41 $14 $51 $47 $61 $114 $111 $121 $58 $9 $126 $114 $24 $44 $86 $44 $34 $49 $126 $116 $119 $435 $128 $4 $125 $7 $57 $79 $104 $31 $118 $82 $125 $4 $25 $66 $157 $76 $44 $46 $88 $96 $124 $674 $118 $124 $101 $89 $84 $67 $111 $21 $104 $127 $16 $120 $45 $95 $35 $114 $116 $79 $201 $132 $114 $181 $23 $64 $55 $28 $113 $84 $98 $44 $6 $126 $133 $130 $123 $118 $124 $95 $129 $295 $114 $105 $94 

In [ ]:
#Generate a performance plot
constant_pricer_tester.report()

The constant price baseline achieved MAE = \$112.21, MSE = 24,842 and R² ≈ -0, outperforming random prediction baseline.

----

In [ ]:
#Extract product weight, missing-weight indicator, and summary length as features for baseline linear regression
def get_features(item):
    return {
        "weight": item.weight,
        "weight_unknown": 1 if item.weight==0 else 0,
        "text_length": len(item.summary)
    }

In [ ]:
def list_to_dataframe(items):
    features = [get_features(item) for item in items]
    df = pd.DataFrame(features)
    df['price'] = [item.price for item in items]
    return df

train_df = list_to_dataframe(train)
test_df = list_to_dataframe(test)

In [ ]:
# Separate features and target
feature_columns = ['weight', 'weight_unknown', 'text_length']

#Continuous features to scale
num_vars = ['weight', 'text_length']
X_train = train_df[feature_columns]
y_train = train_df['price']
X_test = test_df[feature_columns]
y_test = test_df['price']

#Scale continuous features while retaining weight_unknown unchanged
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_vars)
    ],
    remainder='passthrough'
)

#Create linear regression pipeline
model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

#Fit and train the Linear Regression
model.fit(X_train, y_train)
#Get transformed feature names
feature_names = model.named_steps['preprocessor'].get_feature_names_out()
#Access the fitted linear regression estimator
regressor = model.named_steps['regressor']

#Print the coefficients and intercept
for feature, coef in zip(feature_names, regressor.coef_):
    print(f"\n{feature}: {coef:.4}")
print(f"\nIntercept: {regressor.intercept_:.4}")

# Predict the test set and evaluate
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"\nMean Squared Error: {mse:.4f}")
print(f"\nR-squared Score: {r2:.4f}")


num__weight: 19.8

num__text_length: 13.35

remainder__weight_unknown: -7.155

Intercept: 144.9

Mean Squared Error: 24373.6703

R-squared Score: 0.0356


Let's try to understand the coefficients associated with the features and the intercept of the linear regression.

The intercept is the predicted price when both weight and text length features are zero and weight_unknown is zero.

Since weight and text_length are standardized, their coefficients are not interpreted per original unit. The weight_unknown coefficient, however, remains associated with a change from 0 to 1.

The coefficient associated with weight(scaled) is interpretted as follows: A one-standard-deviation increase in weight is associated with a \$19.80 increase in predicted price, holding text length and weight unknow features constant.

For the text_length (scaled) coefficient of \+13.35:	A one-standard-deviation increase in text length is associated with a \$13.35 increase in predicted price, holding the coeffients of weight and weight unknow constant.

For weight_unknown coefficient of \-7.16:	Missing weight is associated with a \$7.16 lower predicted price, holding the coefficients of weight and text length constant.

These coefficients suggest that both product weight and text length are positively associated with price in the training data while a missing weight is associated with a decrease a product price.

However, these coefficients do not establish causation, and their values alone do not tell us how accurately the model predicts product prices.

In [ ]:
def linear_regression_pricer(item):
    features = get_features(item)
    features_df = pd.DataFrame([features])[feature_columns]
    return model.predict(features_df)[0]

In [ ]:
item = test[0]
predicted_price = linear_regression_pricer(item)

print(f"Predicted price: ${predicted_price:,.2f}")
print(f"Actual price: ${item.price:,.2f}")

Predicted price: $133.06
Actual price: $58.73


In [ ]:
linear_regression_pricer_tester = evaluate(linear_regression_pricer, test, size=1000)

  0%|          | 0/1000 [00:00<?, ?it/s]

$74 $117 $236 $75 $5 $825 $102 $55 $31 $120 $124 $174 $84 $90 $400 $114 $89 $76 $135 $33 $40 $410 $28 $84 $47 $293 $110 $44 $122 $97 $62 $89 $128 $19 $29 $259 $71 $219 $144 $21 $17 $294 $63 $105 $101 $82 $72 $79 $39 $130 $131 $160 $348 $11 $135 $118 $55 $122 $126 $58 $80 $57 $57 $125 $112 $53 $115 $125 $70 $84 $137 $21 $77 $100 $111 $132 $38 $101 $122 $99 $102 $74 $28 $51 $17 $100 $128 $63 $82 $117 $123 $108 $86 $121 $93 $86 $343 $74 $97 $78 $138 $45 $17 $99 $38 $22 $113 $97 $14 $52 $58 $44 $77 $84 $405 $373 $113 $6 $265 $96 $2 $317 $113 $20 $7 $90 $76 $221 $122 $122 $34 $129 $124 $73 $103 $132 $74 $121 $48 $70 $26 $51 $23 $48 $101 $104 $128 $72 $5 $121 $110 $40 $50 $91 $56 $56 $58 $121 $118 $124 $427 $129 $11 $112 $26 $74 $66 $113 $26 $94 $71 $130 $10 $12 $79 $144 $92 $55 $52 $84 $105 $137 $667 $106 $120 $97 $83 $94 $66 $107 $1 $110 $110 $2 $126 $41 $80 $40 $121 $130 $77 $220 $129 $104 $192 $2 $79 $52 $33 $99 $96 $97 $40 $8 $124 $134 $126 $125 $119 $142 $95 $129 $273 $95 $126 $94 $102

In [ ]:
#Generate a performance plot
linear_regression_pricer_tester.report()

In [ ]:
#Building a baseline for predicting product prices from their natural language descriptions
#Words in each product's summary is used as an input feature
prices = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]

In [ ]:
np.random.seed(42)
#Create text regression pipeline
nlp_model = Pipeline([
    ('vectorizer', CountVectorizer(max_features=5000, stop_words='english')),#Converts text into numerical features based on word frequencies. Limits each product summary to 2000 words
    ('regressor', LinearRegression())])#Regression algorithm finds coefficients that minimize the sum of squared prediction errors.

nlp_model.fit(documents, prices)

Pipeline(steps=[('vectorizer',
                 CountVectorizer(max_features=5000, stop_words='english')),
                ('regressor', LinearRegression())])

In [ ]:
#Retrieve the vocabulary learned during training
selected_words = nlp_model.named_steps['vectorizer'].get_feature_names_out()
#selected_words = vectorizer.get_feature_names_out()
print(f"Number of selected words: {len(selected_words)}")
print("Selected words:", selected_words[1000:1020])

Number of selected words: 5000
Selected words: ['chalk' 'chalkboard' 'challenge' 'challenger' 'challenges' 'chamber'
 'chandelier' 'change' 'changes' 'changing' 'channel' 'channels'
 'character' 'characters' 'charcoal' 'charge' 'charged' 'charger'
 'chargers' 'charges']


In [ ]:
#Predict product prices using the trained regression model
def natural_language_linear_regression_pricer(item):
    #Transforms new product summaries into numerical vector
    pred = nlp_model.predict([item.summary])[0]
    return max(float(pred), 0)

In [ ]:
natural_language_linear_regression_pricer_tester = evaluate(natural_language_linear_regression_pricer, test, size=1000)

  0%|          | 0/1000 [00:00<?, ?it/s]

$29 $95 $145 $0 $29 $668 $0 $80 $30 $120 $20 $149 $77 $3 $252 $8 $30 $79 $22 $145 $115 $329 $130 $69 $3 $311 $55 $43 $14 $127 $59 $59 $35 $122 $8 $82 $42 $158 $74 $9 $126 $200 $84 $134 $13 $64 $82 $21 $9 $27 $152 $13 $168 $71 $31 $20 $92 $21 $10 $23 $62 $3 $141 $121 $32 $2 $207 $15 $87 $61 $15 $115 $2 $35 $14 $169 $99 $15 $31 $168 $18 $198 $56 $18 $145 $17 $63 $4 $28 $104 $17 $116 $6 $4 $23 $30 $315 $22 $5 $28 $55 $24 $57 $15 $13 $87 $15 $14 $12 $38 $320 $96 $9 $26 $77 $301 $2 $47 $54 $13 $58 $364 $98 $122 $22 $25 $133 $142 $1 $6 $137 $16 $47 $65 $41 $53 $42 $1 $38 $53 $52 $103 $30 $10 $3 $3 $152 $55 $17 $73 $30 $32 $13 $134 $139 $75 $88 $5 $106 $25 $34 $56 $221 $19 $6 $23 $87 $9 $94 $6 $164 $19 $103 $54 $5 $235 $82 $8 $27 $65 $238 $1 $519 $128 $7 $50 $54 $67 $61 $33 $16 $81 $17 $13 $125 $27 $127 $4 $76 $28 $98 $109 $12 $51 $57 $18 $91 $35 $16 $51 $55 $226 $28 $37 $16 $31 $89 $156 $163 $62 $17 $2 $331 $20 $124 $60 $40 $45 $3 $33 $37 $104 $71 $135 $50 $80 $42 $155 $89 $3 $72 $102 $41 $7

In [ ]:
#Generate a performance plot
natural_language_linear_regression_pricer_tester.report()

Random Forest Model

In [ ]:
#Vectorize product summaries and train a Random Forest regressor
#Initialize Randome Forest regressor
rf_model = RandomForestRegressor(n_estimators=100, max_depth=20, min_samples_leaf=2,
                                 max_features=0.1, n_jobs=-1, random_state=42)
#Initialize text vectorizer
rf_vectorizer = CountVectorizer(max_features=4000, stop_words='english')
#Vectorize training product summaries
X = rf_vectorizer.fit_transform(documents)
#Train the Random Forest
rf_model.fit(X, prices)

RandomForestRegressor(max_depth=20, max_features=0.1, min_samples_leaf=2,
                      n_jobs=-1, random_state=42)

In [ ]:
#Predict product prices using the trained Random Forest regressor
def random_forest_pricer(item):
    x = rf_vectorizer.transform([item.summary])
    return max(0, rf_model.predict(x)[0])

In [ ]:
#Evaluate the performance of the model on the first 1000 items in the test set.
random_forest_pricer_tester = evaluate(random_forest_pricer, test, size=1000)

  0%|          | 0/1000 [00:00<?, ?it/s]

$15 $103 $199 $35 $24 $794 $79 $95 $13 $133 $50 $184 $67 $33 $397 $43 $60 $2 $56 $10 $29 $448 $27 $29 $74 $235 $103 $88 $66 $101 $1 $72 $100 $49 $22 $242 $10 $135 $138 $40 $137 $244 $42 $116 $19 $94 $65 $75 $17 $33 $111 $80 $313 $33 $65 $94 $19 $67 $14 $66 $92 $81 $83 $73 $6 $48 $183 $93 $49 $74 $67 $44 $48 $55 $118 $172 $37 $95 $94 $99 $102 $66 $52 $64 $29 $141 $71 $56 $109 $71 $51 $94 $14 $96 $1 $64 $297 $19 $71 $58 $70 $18 $4 $26 $16 $18 $102 $41 $19 $84 $147 $34 $76 $157 $383 $356 $53 $39 $28 $42 $33 $352 $93 $17 $18 $39 $93 $170 $111 $86 $59 $30 $69 $119 $23 $92 $32 $4 $81 $115 $8 $72 $24 $34 $86 $42 $44 $14 $15 $101 $91 $17 $24 $72 $109 $13 $85 $78 $93 $53 $38 $106 $20 $95 $8 $8 $109 $81 $64 $90 $65 $7 $6 $16 $47 $110 $110 $30 $66 $62 $254 $81 $502 $137 $140 $36 $115 $52 $82 $59 $1 $74 $79 $16 $105 $12 $212 $18 $41 $24 $111 $221 $104 $132 $109 $52 $87 $69 $3 $55 $57 $87 $18 $27 $69 $55 $154 $89 $101 $39 $68 $80 $315 $93 $125 $72 $86 $79 $98 $66 $79 $179 $79 $61 $67 $18 $16 $73 $1

In [ ]:
#Generate a performance plot
random_forest_pricer_tester.report()

In [ ]:
#Vectorize product summaries into bag-of-words features and train an XGBoost regressor
np.random.seed(42)
xgb_model = xgb.XGBRegressor(n_estimators=1000, max_depth=10, learning_rate=0.1,
                              tree_method="hist",random_state=42, n_jobs=-1)
vectorizer = CountVectorizer(max_features=4000, stop_words='english')
X = vectorizer.fit_transform(documents)
xgb_model.fit(X, prices)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=True, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=10,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=1000,
             n_jobs=-1, num_parallel_tree=None, ...)

In [ ]:
#Predict product prices using the trained XGBoost regressor
def xg_boost(item):
    x = vectorizer.transform([item.summary])
    return max(0, xgb_model.predict(x)[0])

In [ ]:
#Evaluate the performance of the model on the first 1000 items in the test set.
xg_boost_tester = evaluate(xg_boost, test, size=1000)

  0%|          | 0/1000 [00:00<?, ?it/s]

$18 $79 $200 $4 $29 $591 $18 $79 $19 $106 $34 $151 $47 $2 $132 $22 $14 $5 $6 $97 $97 $70 $99 $15 $55 $279 $97 $3 $8 $59 $78 $7 $68 $48 $3 $1 $51 $85 $67 $29 $141 $47 $27 $141 $8 $56 $32 $57 $19 $15 $57 $6 $135 $32 $3 $12 $28 $30 $11 $59 $107 $26 $102 $131 $23 $1 $121 $1 $63 $43 $13 $48 $32 $12 $60 $168 $108 $11 $37 $121 $9 $77 $15 $54 $47 $110 $56 $15 $100 $82 $17 $132 $11 $39 $3 $38 $318 $39 $18 $23 $34 $48 $56 $31 $1 $46 $16 $30 $21 $47 $170 $30 $36 $82 $113 $289 $5 $44 $25 $29 $36 $228 $70 $58 $46 $22 $46 $161 $26 $2 $104 $23 $39 $62 $20 $64 $51 $9 $44 $102 $66 $58 $2 $24 $12 $32 $63 $25 $5 $41 $11 $14 $15 $80 $152 $41 $89 $60 $168 $0 $73 $41 $77 $19 $27 $1 $89 $30 $52 $0 $183 $10 $105 $4 $1 $65 $108 $2 $6 $51 $187 $21 $123 $160 $2 $151 $32 $12 $86 $17 $5 $59 $6 $61 $100 $11 $201 $13 $72 $8 $63 $29 $12 $47 $72 $50 $31 $36 $8 $32 $37 $79 $24 $34 $14 $19 $78 $135 $108 $35 $3 $16 $362 $32 $148 $33 $13 $19 $54 $18 $14 $173 $87 $105 $31 $15 $41 $138 $99 $27 $77 $77 $10 $85 $8 $60 $14 $32

In [ ]:
#Generate a performance plot
xg_boost_tester.report()

Artificial Neural Network (ANN)

In [ ]:
#Prepare the product summaries and prices
y = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]

In [ ]:
#Use the HashingVectorizer for a Bag of Words model
#Vectorize product summaries into bag-of-words features
np.random.seed(42)
ann_vectorizer = HashingVectorizer(n_features=4000, stop_words='english', binary=True)
X = ann_vectorizer.fit_transform(documents)

In [ ]:
# Define the neural network layers
class NeuralNetwork(nn.Module):
    def __init__(self, input_size):
        super(NeuralNetwork, self).__init__()
        self.layer1 = nn.Linear(input_size, 256)
        self.layer2 = nn.Linear(256, 128)
        self.layer3 = nn.Linear(128, 64)
        self.layer4 = nn.Linear(64, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        output1 = self.relu(self.layer1(x))
        output2 = self.relu(self.layer2(output1))
        output3 = self.relu(self.layer3(output2))
        output4 = self.layer4(output3)
        return output4

In [ ]:
# Convert data to PyTorch tensors
X_train_tensor = torch.FloatTensor(X.toarray())
y_train_tensor = torch.FloatTensor(y).unsqueeze(1)

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_train_tensor, y_train_tensor, test_size=0.01, random_state=42)

# Create the loader
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# Initialize the model
input_size = X_train_tensor.shape[1]
model = NeuralNetwork(input_size)

In [ ]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Number of trainable parameters: {trainable_params:,}")

Number of trainable parameters: 1,065,473


In [ ]:
# Define loss function and optimizer
loss_function = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

#Train the ANN model
EPOCHS = 10 #Perform 10 complete runs through the training data
for epoch in range(EPOCHS):
    model.train()
    for batch_X, batch_y in tqdm(train_loader):
        optimizer.zero_grad()

        # The next 4 lines are the 4 stages of training: forward pass, loss calculation, backward pass, optimize
        outputs = model(batch_X)
        loss = loss_function(outputs, batch_y)
        loss.backward()
        optimizer.step()

    #Validate the trained model on the 10% held-out train set
    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val)
        val_loss = loss_function(val_outputs, y_val)

    print(f"Epoch {epoch+1}/{EPOCHS} | " f"Train Loss: {loss.item():.3f} | " f"Val Loss: {val_loss.item():.3f}")

  0%|          | 0/12375 [00:00<?, ?it/s]

Epoch 1/10 | Train Loss: 6865.512 | Val Loss: 13626.476


  0%|          | 0/12375 [00:00<?, ?it/s]

Epoch 2/10 | Train Loss: 9808.286 | Val Loss: 12593.628


  0%|          | 0/12375 [00:00<?, ?it/s]

Epoch 3/10 | Train Loss: 15845.833 | Val Loss: 11823.669


... (10 outputs left out)

  0%|          | 0/12375 [00:00<?, ?it/s]

Epoch 9/10 | Train Loss: 3148.966 | Val Loss: 12561.106


  0%|          | 0/12375 [00:00<?, ?it/s]

Epoch 10/10 | Train Loss: 2762.447 | Val Loss: 12615.228


In [ ]:
#Predict product prices using the trained ANN model
def neural_network(item):
    model.eval()
    with torch.no_grad():
        vector = ann_vectorizer.transform([item.summary])
        vector = torch.FloatTensor(vector.toarray())
        result = model(vector)[0].item()
    return max(0, result)

In [ ]:
#Evaluate the performance of the ANN model on 1000 items in the test set.
neural_network_tester = evaluate(neural_network, test, size=1000)

  0%|          | 0/1000 [00:00<?, ?it/s]

$59 $60 $173 $26 $24 $295 $9 $32 $49 $44 $5 $112 $66 $0 $51 $32 $22 $105 $24 $12 $4 $264 $130 $67 $15 $102 $75 $44 $9 $80 $25 $17 $17 $41 $58 $68 $42 $230 $220 $21 $17 $127 $39 $65 $77 $174 $98 $40 $65 $16 $54 $2 $111 $36 $17 $6 $15 $19 $2 $64 $118 $98 $7 $49 $14 $31 $150 $2 $49 $93 $33 $51 $199 $2 $106 $60 $139 $3 $51 $17 $28 $86 $77 $61 $15 $29 $40 $10 $78 $23 $8 $29 $1 $22 $3 $13 $233 $16 $4 $36 $18 $9 $60 $32 $16 $49 $17 $116 $39 $30 $324 $12 $8 $34 $6 $372 $8 $81 $14 $28 $46 $320 $71 $36 $41 $6 $61 $130 $34 $4 $121 $18 $10 $0 $26 $28 $51 $9 $28 $121 $120 $31 $50 $28 $4 $4 $40 $41 $18 $7 $7 $7 $38 $95 $112 $6 $189 $111 $21 $5 $22 $17 $102 $9 $41 $36 $137 $56 $15 $24 $98 $12 $50 $8 $2 $91 $46 $35 $126 $129 $70 $2 $104 $37 $13 $47 $8 $120 $106 $67 $48 $3 $6 $22 $62 $17 $27 $104 $34 $9 $17 $238 $18 $0 $147 $41 $10 $74 $50 $10 $48 $39 $16 $17 $57 $8 $36 $116 $76 $20 $6 $1 $301 $28 $123 $32 $22 $4 $21 $5 $4 $291 $72 $55 $17 $11 $3 $62 $105 $36 $54 $83 $24 $59 $12 $33 $23 $42 $33 $28 $86

In [ ]:
#Generate a performance plot
neural_network_tester.report()

Stacked AutoEncoder (SAE)

In [ ]:
#Prepare the dataset
np.random.seed(42)
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#Prepare training documents and prices
train_documents = [item.summary for item in train]
y_train = np.array([float(item.price) for item in train], dtype=np.float32)

#Prepare validation documents and prices
val_documents = [item.summary for item in val]
y_val = np.array([float(item.price) for item in val], dtype=np.float32)

#Vectorize product summaries into binary hashed features
SAE_vectorizer = HashingVectorizer(n_features=4000, stop_words="english", binary=True, dtype=np.float32)
#Transform training and validation documents into numeric representations for the machine learning process.
#Each column feature is one-hot encode into 0s and 1s
X_train = SAE_vectorizer.fit_transform(train_documents)
X_val = SAE_vectorizer.transform(val_documents)

# Convert datasets into PyTorch tensors
X_train_tensor = torch.from_numpy(X_train.toarray())
y_train_tensor = torch.from_numpy(y_train)
X_val_tensor = torch.from_numpy(X_val.toarray())
y_val_tensor = torch.from_numpy(y_val)

#Create batches of hashed features and prices for the SAE pipeline
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

print(f"Device: {device}")
print(f"Training samples: {X_train.shape[0]:,}")
print(f"Validation samples: {X_val.shape[0]:,}")
print(f"Input features: {X_train.shape[1]:,}")

Device: cuda
Training samples: 799,998
Validation samples: 10,000
Input features: 4,000


In [ ]:
#Define the stacked autoencoder (SAE) to reconstruct the 5000 dimensional product text features
class SAE(nn.Module):
    def __init__(self, input_size):#Specifies the number of input size which in this case is 5000 numeric features
        super().__init__()
        #Define the encoder
        self.encoder = nn.Sequential(nn.Linear(input_size, 256), nn.ReLU(),
                                    nn.Linear(256, 128), nn.ReLU(),
                                    nn.Linear(128, 64), nn.ReLU())# Extract 64 dimensional encoded representations
        #Define the decoder
        self.decoder = nn.Sequential(nn.Linear(64, 128), nn.ReLU(),
                                    nn.Linear(128, 256), nn.ReLU(),
                                    nn.Linear(256, input_size))
    #Connect the eNcoder and the decoder
    def forward(self, x):
        return self.decoder(self.encoder(x))

In [ ]:
#Pretrain the autoencoder
auto_encoder = SAE(input_size=X_train_tensor.shape[1]).to(device)
SAE_loss = torch.nn.MSELoss()
optimizer = optim.Adam(auto_encoder.parameters(), lr=0.001)

EPOCHS = 10
for epoch in range(EPOCHS):
    auto_encoder.train()
    train_loss = 0.0

    for batch_X, _ in tqdm(train_loader, desc=f"SAE Epoch {epoch+1}"):
        batch_X = batch_X.to(device)
        optimizer.zero_grad()
        decoded = auto_encoder(batch_X)
        loss = SAE_loss(decoded, batch_X)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * batch_X.size(0)#Training loss for the current batch
    train_loss /= len(train_loader.dataset)#Calculates the average loss per product across the entire batch

    # Validation
    auto_encoder.eval()
    val_loss = 0.0

    with torch.no_grad():
        for batch_X, _ in val_loader:
            batch_X = batch_X.to(device)
            decoded = auto_encoder(batch_X)
            loss = SAE_loss(decoded, batch_X)
            val_loss += loss.item() * batch_X.size(0)
    val_loss /= len(val_loader.dataset)

    print(f"Epoch {epoch+1}/{EPOCHS} | " f"Train SAE MSE: {train_loss:.6f} | " f"Val SAE MSE: {val_loss:.6f}")

SAE Epoch 1:   0%|          | 0/12500 [00:00<?, ?it/s]

Epoch 1/10 | Train SAE MSE: 0.000177 | Val SAE MSE: 0.000167


SAE Epoch 2:   0%|          | 0/12500 [00:00<?, ?it/s]

Epoch 2/10 | Train SAE MSE: 0.000164 | Val SAE MSE: 0.000161


SAE Epoch 3:   0%|          | 0/12500 [00:00<?, ?it/s]

Epoch 3/10 | Train SAE MSE: 0.000160 | Val SAE MSE: 0.000159


... (10 outputs left out)

SAE Epoch 9:   0%|          | 0/12500 [00:00<?, ?it/s]

Epoch 9/10 | Train SAE MSE: 0.000155 | Val SAE MSE: 0.000155


SAE Epoch 10:   0%|          | 0/12500 [00:00<?, ?it/s]

Epoch 10/10 | Train SAE MSE: 0.000155 | Val SAE MSE: 0.000155


In [ ]:
#Define the SAE regressor
#Regressor uses pretrained encoder and add regression head
class SAEPriceRegressor(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.regressor = nn.Sequential(nn.Linear(64, 32), nn.ReLU(),
                         nn.Linear(32, 1))

    def forward(self, x):
        features = self.encoder(x)
        return self.regressor(features).squeeze(-1)

In [ ]:
#Fine-tune the entire encoder and regression head
price_model = SAEPriceRegressor(auto_encoder.encoder).to(device)#auto_encoder.encoder contains the pretrained encoder weights
#Unfreeze the encoder parameters
for param in price_model.parameters():
    param.requires_grad = True

#Define the optimizer parameters for both the encoder and regression head which will both be updated duringg supevised training
optimizer = optim.Adam([
    {"params": price_model.encoder.parameters(), "lr": 0.0001},
    {"params": price_model.regressor.parameters(), "lr": 0.001}
])

In [ ]:
#Train and validate the fine-tuned model
EPOCHS = 20
#Define the loss function
loss_function = torch.nn.MSELoss()
best_val_loss = float("inf")
best_state = None

for epoch in range(EPOCHS):
    price_model.train()
    train_loss = 0.0

    for batch_X, batch_y in tqdm(train_loader, desc=f"Price Epoch {epoch+1}"):
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()
        predictions = price_model(batch_X)
        loss = loss_function(predictions, batch_y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * batch_X.size(0)
    train_loss /= len(train_loader.dataset)

    # Validation
    price_model.eval()
    val_loss = 0.0
    val_mae = 0.0

    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)

            predictions = price_model(batch_X)
            loss = loss_function(predictions, batch_y)

            val_loss += loss.item() * batch_X.size(0)
            val_mae += torch.abs(predictions - batch_y).sum().item()
    val_loss /= len(val_loader.dataset)
    val_mae /= len(val_loader.dataset)

    #Select best model based on validation MSE
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch + 1
        best_state = copy.deepcopy(price_model.state_dict())

    #Report performance after each epoch
    print(f"Epoch {epoch+1}/{EPOCHS} | " f"Train MSE: {train_loss:.2f} | " f"Val MSE: {val_loss:.2f} | " f"Val RMSE: ${math.sqrt(val_loss):.2f} | " f"Val MAE: ${val_mae:.2f}")

#Restore the best model
price_model.load_state_dict(best_state)
price_model.eval()

#Report performance for the best model
print(f"\nBest Epoch: {best_epoch} | " f"Train MSE: {train_loss:.2f} | " f"Val MSE: {best_val_loss:.2f} | " f"Val RMSE: ${math.sqrt(best_val_loss):.2f} | " f"Val MAE: ${val_mae:.2f}")

Price Epoch 1:   0%|          | 0/12500 [00:00<?, ?it/s]

Epoch 1/20 | Train MSE: 16703.90 | Val MSE: 14312.98 | Val RMSE: $119.64 | Val MAE: $73.57


Price Epoch 2:   0%|          | 0/12500 [00:00<?, ?it/s]

Epoch 2/20 | Train MSE: 13445.94 | Val MSE: 12984.73 | Val RMSE: $113.95 | Val MAE: $73.07


Price Epoch 3:   0%|          | 0/12500 [00:00<?, ?it/s]

Epoch 3/20 | Train MSE: 11811.54 | Val MSE: 12199.83 | Val RMSE: $110.45 | Val MAE: $67.84


... (30 outputs left out)

Price Epoch 19:   0%|          | 0/12500 [00:00<?, ?it/s]

Epoch 19/20 | Train MSE: 3020.95 | Val MSE: 13262.31 | Val RMSE: $115.16 | Val MAE: $67.36


Price Epoch 20:   0%|          | 0/12500 [00:00<?, ?it/s]

Epoch 20/20 | Train MSE: 2860.90 | Val MSE: 12817.40 | Val RMSE: $113.21 | Val MAE: $66.71

Best Epoch: 7 | Train MSE: 2860.90 | Val MSE: 11463.96 | Val RMSE: $107.07 | Val MAE: $66.71


In [ ]:
#Predict product prices using the selected trained model
def SAE_pricer(item):
    price_model.eval()
    with torch.no_grad():
        vector = SAE_vectorizer.transform([item.summary])
        vector = torch.from_numpy(vector.toarray()).to(device)
        prediction = price_model(vector)[0].item()
    return max(0.0, prediction)

In [ ]:
#Evaluate the performance of the selected model on the heald-out test set
SAE_pricer_tester = evaluate(SAE_pricer, test, size=1000)

  0%|          | 0/1000 [00:00<?, ?it/s]

$5 $55 $119 $17 $61 $602 $13 $21 $35 $90 $30 $196 $20 $3 $222 $45 $15 $13 $28 $5 $118 $51 $85 $37 $118 $205 $73 $48 $14 $97 $58 $6 $15 $65 $34 $92 $50 $115 $97 $43 $5 $197 $0 $117 $12 $2 $13 $32 $28 $14 $56 $3 $74 $92 $73 $11 $11 $12 $4 $9 $114 $19 $44 $37 $6 $20 $150 $5 $26 $38 $30 $64 $21 $5 $126 $94 $92 $2 $26 $68 $42 $89 $80 $213 $17 $11 $42 $28 $87 $71 $1 $40 $4 $2 $7 $7 $301 $35 $5 $30 $19 $6 $46 $49 $8 $0 $31 $56 $23 $40 $62 $4 $2 $93 $201 $166 $26 $43 $27 $60 $15 $365 $50 $39 $14 $18 $28 $173 $29 $1 $28 $34 $37 $3 $24 $48 $63 $5 $76 $111 $62 $138 $26 $22 $11 $7 $69 $22 $41 $9 $4 $13 $27 $2 $93 $18 $65 $125 $18 $3 $59 $18 $136 $12 $198 $60 $104 $18 $27 $30 $197 $2 $3 $24 $1 $142 $114 $28 $132 $64 $185 $18 $175 $85 $17 $60 $3 $63 $128 $16 $57 $12 $5 $50 $35 $116 $73 $105 $56 $4 $43 $92 $16 $36 $121 $28 $54 $48 $55 $11 $35 $28 $21 $8 $29 $7 $80 $118 $61 $30 $10 $1 $296 $40 $109 $5 $25 $23 $39 $34 $1 $191 $53 $100 $17 $71 $24 $184 $70 $28 $145 $77 $38 $44 $11 $6 $26 $6 $181 $16 $15

In [36]:
import importlib
import pricer.evaluator
importlib.reload(pricer.evaluator)

<module 'pricer.evaluator' from '/content/ProductPriceAI/pricer/evaluator.py'>

In [37]:
#Generate performance evaluation plot
SAE_pricer_tester.report()